# Kannada NER: CRF with Comprehensive Feature Engineering


## Feature Categories Implemented
| Category | Features |
|---|---|
| **Lexical** | Word identity, word shape, prefixes/suffixes (1-5 chars), script type (Kannada / Latin / Mixed) |
| **Orthographic** | All-caps Latin (BBMP, BDA), digit-in-word (5ನೇ), punctuation flags, hyphen, mixed script |
| **Syntactic (POS)** | **Polyglot** POS tags for Kannada (NOUN, PROPN, VERB, ADP...) + context window ±2 |
| **Syntactic (Dependency proxy)** | Relative sentence position, head-candidate heuristic, is-post-verb |
| **Domain-specific (Gazetteer)** | Karnataka districts, taluks, location keywords, org keywords, person titles & honorifics |

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Polyglot: lightweight multilingual NLP library with Kannada POS support
# indic-nlp-library: morphological analysis for Dravidian languages
!pip install polyglot pyicu morfessor pycld2 -q
!pip install sklearn-crfsuite joblib tqdm scikit-learn indic-nlp-library sympy --upgrade -q
!pip install datasets==2.16.0


# Download Polyglot language data for Kannada
!python -c "import polyglot; from polyglot.downloader import downloader; downloader.download('pos2.kn')"

print("Installation complete.")

  Using cached datasets-2.16.0-py3-none-any.whl.metadata (20 kB)
Using cached datasets-2.16.0-py3-none-any.whl (507 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.4
    Uninstalling datasets-4.8.4:
      Successfully uninstalled datasets-4.8.4
[polyglot_data] Error loading pos2.kn: HTTP Error 403: Forbidden
Installation complete.


In [ ]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import re
import unicodedata
from polyglot.text import Text
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.metrics import accuracy_score
from datasets import load_dataset
import joblib
from tqdm.auto import tqdm
# Polyglot import with graceful fallback if installation failed
try:
    from polyglot.text import Text as PolyText
    POLYGLOT_AVAILABLE = True
    print("Polyglot available.")
except ImportError:
    POLYGLOT_AVAILABLE = False
    print("WARNING: Polyglot not available — using morphological fallback POS only.")

Polyglot available.


In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────
TRAIN_SAMPLES = 10000   # 5k balances quality with Polyglot preprocessing time
BATCH_SIZE    = 200     # Sentences processed per Polyglot call

TAG_MAPPING = {
    0: 'O',
    1: 'B-PER', 2: 'I-PER',
    3: 'B-ORG', 4: 'I-ORG',
    5: 'B-LOC', 6: 'I-LOC'
}

In [ ]:
# Cell 4 — Gazetteers & Linguistic Lists

HONORIFICS = {
    'ಡಾ', 'ಡಾ.', 'ಶ್ರೀ', 'ಶ್ರೀಮತಿ', 'ಶ್ರೀಯುತ',
    'ಅವರು', 'ಅವರನ್ನು', 'ಅವರಿಗೆ', 'ಅವರಿಂದ',
    'ಅಣ್ಣ', 'ಅಕ್ಕ', 'ಮಹೋದಯ', 'ಮಹೋದಯರು',
    'ಗೌಡ', 'ಗೌಡರು', 'ರೆಡ್ಡಿ', 'ನಾಯಕ', 'ಶೆಟ್ಟಿ',
    'ಕುಮಾರ', 'ಕುಮಾರಿ', 'ಮಿಸ್', 'ಮಿಸ್ಟರ್', 'ಮಿಸ್ಸೆಸ್'
}

PERSON_POST_TITLES = {
    'ಅಧ್ಯಕ್ಷ', 'ಮುಖ್ಯಮಂತ್ರಿ', 'ಸಚಿವ', 'ನ್ಯಾಯಮೂರ್ತಿ',
    'ಆಯುಕ್ತ', 'ಅಧಿಕಾರಿ', 'ನಿರ್ದೇಶಕ', 'ಮಂತ್ರಿ'
}


LOCATION_TYPE_WORDS = {
    'ನಗರ', 'ಮಹಾನಗರ', 'ಪಟ್ಟಣ', 'ಹಳ್ಳಿ', 'ಗ್ರಾಮ',
    'ಜಿಲ್ಲೆ', 'ತಾಲ್ಲೂಕು', 'ಹೋಬಳಿ',
    'ಬಡಾವಣೆ', 'ಲೇಔಟ್', 'ಬ್ಲಾಕ್', 'ಕ್ರಾಸ್',
    'ರಸ್ತೆ', 'ರೋಡ್', 'ಮಾರ್ಗ',
    'ರಾಜ್ಯ', 'ದೇಶ', 'ನದಿ', 'ಕೆರೆ', 'ಬೆಟ್ಟ'
}

ORG_KEYWORDS = {
    'ಸರ್ಕಾರ', 'ಪ್ರಾಧಿಕಾರ', 'ಆಯೋಗ', 'ಇಲಾಖೆ', 'ನ್ಯಾಯಾಲಯ',
    'ನಿಗಮ', 'ಮಂಡಳಿ', 'ಸಮಿತಿ', 'ಪಾಲಿಕೆ',
    'ಬ್ಯಾಂಕ್', 'ಆಸ್ಪತ್ರೆ', 'ಶಾಲೆ', 'ಕಾಲೇಜು', 'ವಿಶ್ವವಿದ್ಯಾಲಯ',
    'ಸಂಸ್ಥೆ', 'ಕಂಪನಿ', 'ಕಚೇರಿ', 'ಠಾಣೆ'
}

VIBHAKTI_SUFFIXES = [
    'ನಲ್ಲಿ', 'ರಲ್ಲಿ', 'ಕ್ಕೆ', 'ಗೆ',
    'ರಿಂದ', 'ಇಂದ', 'ನ್ನು', 'ಅನ್ನು',
    'ವರೆಗೆ', 'ನ', 'ದ'
]

# Comprehensive Kannada verb suffixes
# Covers: present/past continuous, habitual, imperative, negative
# FIX: Added ದ್ದಾರೆ so 'ಮಾಡುತ್ತಿದ್ದಾರೆ' correctly gets tagged VERB
VERB_SUFFIXES = [
    # Present continuous: ತ್ತಿ + person marker
    'ತ್ತಿದ್ದಾರೆ', 'ತ್ತಾರೆ', 'ತ್ತಾನೆ', 'ತ್ತಾಳೆ', 'ತ್ತೇನೆ', 'ತ್ತೇವೆ',
    'ತ್ತಿದ್ದಾನೆ', 'ತ್ತಿದ್ದಾಳೆ', 'ತ್ತಿದ್ದೇನೆ',
    # Past progressive: ದ್ದ + person marker
    'ದ್ದಾರೆ',   # e.g. ಮಾಡುತ್ತಿದ್ದಾರೆ
    'ದ್ದಾನೆ', 'ದ್ದಾಳೆ', 'ದ್ದೇನೆ', 'ದ್ದೆವೆ', 'ದ್ದರು',
    # Simple past / perfect
    'ಮಾಡಿದರು', 'ಮಾಡಿದ', 'ಮಾಡಿದಳು', 'ಮಾಡಿದನು',
    'ಆಗಿದೆ', 'ಆಗಿದ್ದರು', 'ಹಾಳಾಗಿದೆ',
    # Negative
    'ಬಂದಿಲ್ಲ', 'ಬರುತ್ತಿಲ್ಲ', 'ಆಗಿಲ್ಲ', 'ಮಾಡಿಲ್ಲ',
    # Imperative
    'ಮಾಡಿ', 'ಕೊಡಿ', 'ಕೇಳಿ', 'ಒದಗಿಸಿ', 'ಸರಿಪಡಿಸಿ'
]

print(f"Gazetteers ready. VERB_SUFFIXES: {len(VERB_SUFFIXES)} entries.")

Gazetteers ready. VERB_SUFFIXES: 31 entries.


In [ ]:
# ── Cell 5: Script & Orthographic Utility Functions ───────────────────────────



def has_digit_in_word(word):
    """e.g. '5ನೇ' (5th), '3ನೇ ಬ್ಲಾಕ್' — ordinal position often follows LOC."""
    return any(c.isdigit() for c in word)

def has_vibhakti(word):
    return any(word.endswith(v) for v in VIBHAKTI_SUFFIXES)

def is_verb_like(word):
    """Orthographic proxy for verbs — not POS, but helps distinguish from entities."""
    return any(word.endswith(v) for v in VERB_SUFFIXES)

def is_head_candidate(word):
    """Dependency proxy: words without vibhakti are more likely syntactic heads."""
    return not has_vibhakti(word) and not is_verb_like(word)


KANNADA_RANGE = (0x0C80, 0x0CFF)

def is_kannada_char(ch):
    return KANNADA_RANGE[0] <= ord(ch) <= KANNADA_RANGE[1]

def script_type(word):
    """Classify word's script: KANNADA, LATIN, MIXED, DIGIT, PUNCT."""
    if word.isdigit():  return 'DIGIT'
    chars = [c for c in word if c.isalpha()]
    if not chars:       return 'PUNCT'
    kn = sum(1 for c in chars if is_kannada_char(c))
    la = sum(1 for c in chars if c.isascii())
    if kn == len(chars): return 'KANNADA'
    if la == len(chars): return 'LATIN'
    return 'MIXED'

def word_shape(word):
    """Detailed word shape for orthographic features."""
    alpha_chars = [c for c in word if c.isalpha()]
    if not alpha_chars: return 'NONALPHA'
    # All-caps Latin (acronyms/orgs: BBMP, BDA, BJP)
    if all(c.isupper() and c.isascii() for c in alpha_chars): return 'ALLCAPS_LATIN'
    # Title-case Latin (English names: Ramesh, Kumar)
    if alpha_chars[0].isupper() and alpha_chars[0].isascii(): return 'TITLE_LATIN'
    # Pure Kannada — shape by length
    if all(is_kannada_char(c) for c in alpha_chars):
        if len(word) <= 2:  return 'KN_SHORT'
        if len(word) >= 12: return 'KN_VERYLONG'
        if len(word) >= 7:  return 'KN_LONG'
        return 'KN_MEDIUM'
    return 'MIXED_SCRIPT'

print("Orthographic utilities ready.")

Orthographic utilities ready.


In [ ]:
# ── Cell 6: POS Tagger — Polyglot with Positional Alignment + Morphological Fallback ──
#
# ROOT CAUSE of the 'X' bug:
#   Polyglot re-tokenizes the input string internally, producing its own token list.
#   The dict lookup `pos_map.get(tok, 'X')` fails because Polyglot's tokenization
#   rarely matches our Naamapadam tokens exactly → every lookup returns 'X'.
#
# FIX:
#   Step 1 — Use POSITIONAL alignment: zip Polyglot's output by index, not by token string.
#   Step 2 — If Polyglot returns a different number of tokens (common), fall back to
#             our morphological rule-based POS tagger which ALWAYS works.

def morph_pos(word):
    """
    Rule-based morphological POS tagger for Kannada.
    """
    if not word: return 'X'
    if word.isdigit(): return 'NUM'
    # All-caps Latin → acronym / org name (BBMP, BDA, BJP)
    if all(c.isupper() for c in word if c.isalpha()) and any(c.isascii() for c in word):
        return 'PROPN'
    # Title-case Latin → English proper noun (Ramesh, Kumar)
    if word[0].isupper() and word[0].isascii():
        return 'PROPN'
    # Verb suffix match
    if is_verb_like(word): return 'VERB'
    # Vibhakti (case marker) suffix — the word IS inflected, likely NOUN form
    if has_vibhakti(word): return 'NOUN'
    # Honorific → PART (particle)
    if word in HONORIFICS: return 'PART'

    # Kannada-only words: use length as a crude PROPN vs NOUN heuristic
    alpha = [c for c in word if c.isalpha()]
    if alpha and all(is_kannada_char(c) for c in alpha):
        # Shorter, bare Kannada words tend to be nouns / function words
        return 'NOUN'

    return 'X'


def get_pos_tags(token_list):
    """
    Attempt Polyglot POS tagging with positional index alignment.
    Falls back to morph_pos() per token if:
      - Polyglot is unavailable
      - Polyglot's token count doesn't match ours (common for Kannada)
      - Any exception is raised
    """
    if POLYGLOT_AVAILABLE:
        try:
            sentence = " ".join(token_list)
            pg_tags  = PolyText(sentence, hint_language_code='kn').pos_tags
            # pg_tags: list of (word_str, tag_str) from Polyglot's own tokenization

            if len(pg_tags) == len(token_list):
                # Perfect alignment — use Polyglot tags directly
                return [tag for _, tag in pg_tags]

            # Misalignment: Polyglot split differently → fall back to morph
            # (This is the common case for long agglutinated Kannada words)
        except Exception:
            pass  # Fall through to morphological fallback

    # Morphological fallback: always produces a valid tag for every token
    return [morph_pos(tok) for tok in token_list]


# ── Sanity check ──────────────────────────────────────────────────────────────
test_tokens = ['ಬೆಂಗಳೂರಿನಲ್ಲಿ', 'ರಮೇಶ್', 'ಅವರು', 'ಕೆಲಸ', 'ಮಾಡುತ್ತಿದ್ದಾರೆ', 'BBMP']
test_pos    = get_pos_tags(test_tokens)
print(f"  {'Token':<25} POS")
print(f"  {'-'*38}")
for tok, pos in zip(test_tokens, test_pos):
    print(f"  {tok:<25} {pos}")

  Token                     POS
  --------------------------------------
  ಬೆಂಗಳೂರಿನಲ್ಲಿ             NOUN
  ರಮೇಶ್                     NOUN
  ಅವರು                      PART
  ಕೆಲಸ                      NOUN
  ಮಾಡುತ್ತಿದ್ದಾರೆ            VERB
  BBMP                      PROPN


In [ ]:
# ── Cell 7: Load Naamapadam Dataset ───────────────────────────────────────────

print("Loading Naamapadam Kannada dataset...")
dataset = load_dataset("ai4bharat/naamapadam", "kn")
train_data = dataset['train'].select(range(TRAIN_SAMPLES))
print(f"Loaded {len(train_data)} sentences.")

Loading Naamapadam Kannada dataset...


Generating train split:   0%|          | 0/471763 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1019 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2381 [00:00<?, ? examples/s]

Loaded 50000 sentences.


In [ ]:
# Cell 8 — Batch POS Tagging
print(f"POS-tagging {TRAIN_SAMPLES} sentences...")

all_tokens, all_pos, all_ner = [], [], []

for i in tqdm(range(TRAIN_SAMPLES), desc="POS"):
    row    = train_data[i]
    tokens = row['tokens']
    tags   = [TAG_MAPPING[t] for t in row['ner_tags']]
    pos    = get_pos_tags(tokens)
    all_tokens.append(tokens)
    all_pos.append(pos)
    all_ner.append(tags)

# Quality check
verb_hits = sum(1 for pos_seq in all_pos for p in pos_seq if p == 'VERB')
x_hits    = sum(1 for pos_seq in all_pos for p in pos_seq if p == 'X')
total     = sum(len(s) for s in all_pos)
print(f"\nDone. VERB rate: {verb_hits/total:.1%}  |  Unknown (X) rate: {x_hits/total:.1%}")

POS-tagging 10000 sentences...


POS:   0%|          | 0/10000 [00:00<?, ?it/s]


Done. VERB rate: 2.3%  |  Unknown (X) rate: 0.2%


In [ ]:
# ── Cell 9: Sanity Check ──────────────────────────────────────────────────────
print("Sample sentence with all annotation layers:\n")
print(f"  {'Token':<22} {'POS':<10} {'Shape':<16} {'NER'}")
print(f"  {'-'*65}")
for tok, pos, ner in zip(all_tokens[0], all_pos[0], all_ner[0]):
    print(f"  {tok:<22} {pos:<10} {word_shape(tok):<16} {ner}")

Sample sentence with all annotation layers:

  Token                  POS        Shape            NER
  -----------------------------------------------------------------
  ನಿರ್ದೇಶಕರುಃ            NOUN       KN_LONG          O
  ಶಿವಗಣೇಶ್               NOUN       KN_LONG          B-PER


In [ ]:
# ── Cell 10: Full word2features Function ──────────────────────────────────────
# Covers all 4 standard NER feature categories

def word2features(sent_tokens, sent_pos, i, sent_len):
    word = sent_tokens[i]
    pos  = sent_pos[i]

    features = {
        # ================================================================
        # 1. LEXICAL FEATURES
        # ================================================================
        'bias': 1.0,
        'word': word,
        'word.lower()': word.lower(),

        # Suffixes (capture vibhaktis: ನಲ್ಲಿ, ರಿಂದ, ಗೆ...)
        'suf1': word[-1:]  if len(word) >= 1 else '',
        'suf2': word[-2:]  if len(word) >= 2 else word,
        'suf3': word[-3:]  if len(word) >= 3 else word,
        'suf4': word[-4:]  if len(word) >= 4 else word,
        'suf5': word[-5:]  if len(word) >= 5 else word,

        # Prefixes (capture titles: ಶ್ರೀ, ಡಾ...)
        'pre2': word[:2]   if len(word) >= 2 else word,
        'pre3': word[:3]   if len(word) >= 3 else word,

        'word.length': len(word),
        'word.script': script_type(word),

        # ================================================================
        # 2. ORTHOGRAPHIC FEATURES
        # ================================================================
        'word.shape':          word_shape(word),
        'word.isdigit':        word.isdigit(),
        'word.has_digit':      has_digit_in_word(word),     # e.g. 5ನೇ
        'word.is_allcaps_lat': word_shape(word) == 'ALLCAPS_LATIN',  # BBMP, BJP
        'word.is_title_lat':   word_shape(word) == 'TITLE_LATIN',    # Ramesh
        'word.is_mixed':       script_type(word) == 'MIXED',
        'word.has_hyphen':     '-' in word or '–' in word,
        'word.has_punct':      any(c in word for c in '.,!?;:'),

        # ================================================================
        # 3a. SYNTACTIC FEATURES — POS (Polyglot)
        # ================================================================
        'pos':          pos,
        'is_propn':     pos == 'PROPN',    # Strong entity signal
        'is_noun':      pos == 'NOUN',
        'is_verb':      pos == 'VERB',
        'is_adp':       pos == 'ADP',      # Postposition → previous word is entity
        'is_num':       pos == 'NUM',

        # ================================================================
        # 3b. SYNTACTIC FEATURES — Dependency Proxy
        # (No full parser: use position & morphology heuristics)
        # ================================================================
        'dep.rel_position':    round(i / max(1, sent_len - 1), 2),  # 0.0=start, 1.0=end
        'dep.is_sent_initial': (i == 0),                # Sentence-initial PROPN is often entity
        'dep.is_sent_final':   (i == sent_len - 1),
        'dep.has_vibhakti':    has_vibhakti(word),      # Has case marker → inflected (not bare entity)
        'dep.is_head_cand':    is_head_candidate(word), # Likely syntactic head
        'dep.is_verb_like':    is_verb_like(word),      # Verb-like → not entity

        # ================================================================
        # 4. DOMAIN-SPECIFIC / GAZETTEER FEATURES
        # ================================================================
        'gaz.is_honorific':      word in HONORIFICS,
        'gaz.is_post_title':     word in PERSON_POST_TITLES,
        'gaz.is_kn_place':       word in KARNATAKA_PLACES,
        'gaz.has_loc_keyword':   any(loc in word for loc in LOCATION_TYPE_WORDS),
        'gaz.has_org_keyword':   any(org in word for org in ORG_KEYWORDS),
        # substring search for multi-word gazetteers
        'gaz.is_loc_type_word':  word in LOCATION_TYPE_WORDS,
        'gaz.is_org_word':       word in ORG_KEYWORDS,
    }

    # ── Context: previous 2 words ────────────────────────────────────────────
    if i >= 2:
        w2, p2 = sent_tokens[i-2], sent_pos[i-2]
        features.update({
            '-2:word':        w2,
            '-2:pos':         p2,
            '-2:is_propn':    (p2 == 'PROPN'),
            '-2:is_honorific': w2 in HONORIFICS,
            '-2:shape':       word_shape(w2),
        })

    if i >= 1:
        w1, p1 = sent_tokens[i-1], sent_pos[i-1]
        features.update({
            '-1:word':        w1,
            '-1:pos':         p1,
            '-1:is_propn':    (p1 == 'PROPN'),
            '-1:is_honorific': w1 in HONORIFICS,
            '-1:is_loc_type': w1 in LOCATION_TYPE_WORDS,
            '-1:is_org_word': w1 in ORG_KEYWORDS,
            '-1:has_vibhakti': has_vibhakti(w1),
            '-1:shape':       word_shape(w1),
        })
    else:
        features['BOS'] = True

    # ── Context: next 2 words ────────────────────────────────────────────────
    if i < sent_len - 1:
        w1, p1 = sent_tokens[i+1], sent_pos[i+1]
        features.update({
            '+1:word':        w1,
            '+1:pos':         p1,
            '+1:is_propn':    (p1 == 'PROPN'),
            '+1:is_honorific': w1 in HONORIFICS,
            # If next word is ADP, current is likely bare-form entity end
            '+1:is_adp':      (p1 == 'ADP'),
            '+1:has_vibhakti': has_vibhakti(w1),
            '+1:is_loc_type': w1 in LOCATION_TYPE_WORDS,
            '+1:is_org_word': w1 in ORG_KEYWORDS,
            '+1:shape':       word_shape(w1),
        })
    else:
        features['EOS'] = True

    if i < sent_len - 2:
        w2, p2 = sent_tokens[i+2], sent_pos[i+2]
        features.update({
            '+2:word':        w2,
            '+2:pos':         p2,
            '+2:is_propn':    (p2 == 'PROPN'),
            '+2:is_post_title': w2 in PERSON_POST_TITLES,
        })

    return features


def sent2features(tokens, pos_tags):
    n = len(tokens)
    return [word2features(tokens, pos_tags, i, n) for i in range(n)]


print(f"word2features defined with {len(word2features(['ಟೆಸ್ಟ್'], ['NOUN'], 0, 1))} features per token.")

word2features defined with 41 features per token.


In [ ]:
# ── Cell 11: Build Feature Matrices ───────────────────────────────────────────
print("Building feature matrices...")
X = [sent2features(all_tokens[i], all_pos[i]) for i in tqdm(range(len(all_tokens)), desc="Features")]
y = all_ner
print(f"Done. {len(X)} feature vectors built.")

Building feature matrices...


Features:   0%|          | 0/10000 [00:00<?, ?it/s]

Done. 10000 feature vectors built.


In [ ]:
# ── Cell 12: Train / Test Split ───────────────────────────────────────────────
split = int(len(X) * 0.9)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 9000 | Test: 1000


In [ ]:
# ── Cell 13: Train CRF ────────────────────────────────────────────────────────
print("Training CRF (L-BFGS, ~2-5 min)...")

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,              # L1 regularisation
    c2=0.1,              # L2 regularisation
    max_iterations=150,
    all_possible_transitions=True,
    verbose=True
)
crf.fit(X_train, y_train)
print("\n✅ Training complete!")

Training CRF (L-BFGS, ~2-5 min)...


loading training data to CRFsuite: 100%|██████████| 9000/9000 [00:06<00:00, 1296.36it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 235608
Seconds required: 1.176

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 150
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=0.54  loss=167014.48 active=234242 feature_norm=1.00
Iter 2   time=0.28  loss=128414.80 active=224885 feature_norm=0.74
Iter 3   time=0.30  loss=116519.94 active=167868 feature_norm=0.96
Iter 4   time=0.29  loss=93201.63 active=234522 feature_norm=1.06
Iter 5   time=0.28  loss=83748.09 active=235516 feature_norm=1.14
Iter 6   time=0.28  loss=82092.31 active=222464 feature_norm=1.33
Iter 7   time=0.31  loss=79383.52 active=231296 feature_norm=1.35
Iter 8   time=0.28  loss=78365.86 active=225964 feature_norm=1.45
Iter 9   time=0.27  loss=71046.61 active=219286 feature_norm=2.28
I

In [ ]:
# ── Cell 14: Evaluation ───────────────────────────────────────────────────────
y_pred = crf.predict(X_test)

labels = [l for l in crf.classes_ if l != 'O']
print("\n=== Per-Entity Classification Report ===")
print(metrics.flat_classification_report(y_test, y_pred, labels=labels, digits=4))

# Overall token accuracy
flat_true = [t for sent in y_test for t in sent]
flat_pred = [t for sent in y_pred for t in sent]
print(f"Overall token accuracy (incl. O): {accuracy_score(flat_true, flat_pred):.4f}")


=== Per-Entity Classification Report ===
              precision    recall  f1-score   support

       B-PER     0.7968    0.6879    0.7383       644
       B-ORG     0.7366    0.5676    0.6412       340
       I-ORG     0.5448    0.4000    0.4613       365
       B-LOC     0.7640    0.6590    0.7077       393
       I-PER     0.7465    0.7840    0.7648       537
       I-LOC     0.4754    0.3494    0.4028        83

   micro avg     0.7273    0.6312    0.6759      2362
   macro avg     0.6773    0.5747    0.6193      2362
weighted avg     0.7210    0.6312    0.6707      2362

Overall token accuracy (incl. O): 0.8775


In [ ]:
# ── Cell 15: Top Learned Features (Interpretability) ──────────────────────────
print("=== Top 15 state features for each entity type ===\n")
for label in ['B-PER', 'B-LOC', 'B-ORG']:
    weights = {attr: w for (attr, lbl), w in crf.state_features_.items()
               if lbl == label and w > 0}
    print(f"── {label} ──")
    for feat, w in sorted(weights.items(), key=lambda x: -x[1])[:15]:
        print(f"  {feat:45s}  {w:+.4f}")
    print()

=== Top 15 state features for each entity type ===

── B-PER ──
  +1:word:ರಾಮಪಟ್ಟಾಭಿಷೇಕ                          +4.3746
  +1:word:ವಿಶ್ಲೇಷಿಸಿದ್ದಾರೆ                       +3.7869
  +2:word:ದಂಪತಿಗಳಿಗೆ                             +3.0196
  -1:word:ಪತ್ನಿ                                  +2.9697
  -1:word:ಮನೀಷ್                                  +2.9696
  -1:word:ಪುತ್ರ                                  +2.9092
  +2:word:ಶಿವರಾತ್ರಿಿದೇಶಿಕೇಂದ್ರ                   +2.7255
  pre3:ಯೇಸ                                       +2.7132
  +2:word:ಲೆ                                     +2.6842
  +2:word:ಡೆವಿಸ್                                 +2.6815
  +1:word:ಬಳ್ಳಾರಿಯವನು                            +2.6692
  -2:word:ಫ್ಲ್ಯಾಟ್ಗಳು                            +2.6547
  +2:word:7                                      +2.5959
  +1:word:ಇಂದಿರಾನಗರದವಳು                          +2.5627
  -1:word:ನವದೆಹಲಿಯ                               +2.5285

── B-LOC ──
  +1:word:ಸಿರಿಯ                                  +3.6031
  pre3:ಪಾಕ 

In [ ]:
# ── Cell 16: Save Model ───────────────────────────────────────────────────────
joblib.dump(crf, 'crf_extractor.pkl')
print("✅ Saved: crf_extractor.pkl  (compatible with existing inference.py)")

✅ Saved: crf_extractor.pkl  (compatible with existing inference.py)


In [ ]:
# ── Cell 17: Quick Inference ──────────────────────────────────────────────────
def predict_ner(text):
    tokens = text.split()
    pos    = get_pos_tags(tokens)
    X_sent = [sent2features(tokens, pos)]
    preds  = crf.predict(X_sent)[0]

    print(f"\nInput : {text}")
    print(f"\n{'Token':<22} {'POS':<8} {'Shape':<16} {'Predicted NER'}")
    print('-' * 70)
    for tok, p, tag in zip(tokens, pos, preds):
        flag = ' ◀ ENTITY' if tag != 'O' else ''
        print(f"{tok:<22} {p:<8} {word_shape(tok):<16} {tag}{flag}")

predict_ner("ಮೈಸೂರಿನಲ್ಲಿ ಲಕ್ಷ್ಮಿ ಅವರ ಬ್ಯಾಂಕ್ ಖಾತೆಯಿಂದ ಹಣ ವರ್ಗಾವಣೆಯಾಗಿದೆ")
print()
predict_ner("ಬೆಂಗಳೂರು ಮಹಾನಗರ ಪಾಲಿಕೆ ಅಧ್ಯಕ್ಷ ರಮೇಶ್ ಕುಮಾರ್ ಅವರು ಹೇಳಿದರು")
print()
predict_ner("BBMP ಕಚೇರಿಯಲ್ಲಿ ಡಾ ಶ್ರೀಧರ್ ಅವರು 4ನೇ ಬ್ಲಾಕ್ ರಸ್ತೆ ದುರಸ್ತಿ ಬಗ್ಗೆ ಮಾತಾಡಿದರು")


Input : ಮೈಸೂರಿನಲ್ಲಿ ಲಕ್ಷ್ಮಿ ಅವರ ಬ್ಯಾಂಕ್ ಖಾತೆಯಿಂದ ಹಣ ವರ್ಗಾವಣೆಯಾಗಿದೆ

Token                  POS      Shape            Predicted NER
----------------------------------------------------------------------
ಮೈಸೂರಿನಲ್ಲಿ            NOUN     KN_LONG          O
ಲಕ್ಷ್ಮಿ                NOUN     KN_LONG          B-PER ◀ ENTITY
ಅವರ                    NOUN     KN_MEDIUM        O
ಬ್ಯಾಂಕ್                NOUN     KN_LONG          O
ಖಾತೆಯಿಂದ               NOUN     KN_LONG          O
ಹಣ                     NOUN     KN_SHORT         O
ವರ್ಗಾವಣೆಯಾಗಿದೆ         NOUN     KN_VERYLONG      O


Input : ಬೆಂಗಳೂರು ಮಹಾನಗರ ಪಾಲಿಕೆ ಅಧ್ಯಕ್ಷ ರಮೇಶ್ ಕುಮಾರ್ ಅವರು ಹೇಳಿದರು

Token                  POS      Shape            Predicted NER
----------------------------------------------------------------------
ಬೆಂಗಳೂರು               NOUN     KN_LONG          B-LOC ◀ ENTITY
ಮಹಾನಗರ                 NOUN     KN_MEDIUM        I-LOC ◀ ENTITY
ಪಾಲಿಕೆ                 NOUN     KN_MEDIUM        O
ಅಧ್ಯಕ್ಷ                NOUN     KN_LONG       

In [ ]:
predict_ner("ರಮೇಶ್ ಅವರು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಇರುವ ಬಿಬಿಎಂಪಿ ಕಚೇರಿಗೆ ಹೋಗಿ ಸಚಿವರನ್ನು ಭೇಟಿ ಮಾಡಿದರು")
print()


Input : ರಮೇಶ್ ಅವರು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಇರುವ ಬಿಬಿಎಂಪಿ ಕಚೇರಿಗೆ ಹೋಗಿ ಸಚಿವರನ್ನು ಭೇಟಿ ಮಾಡಿದರು

Token                  POS      Shape            Predicted NER
----------------------------------------------------------------------
ರಮೇಶ್                  NOUN     KN_MEDIUM        B-PER ◀ ENTITY
ಅವರು                   PART     KN_MEDIUM        O
ಬೆಂಗಳೂರಿನಲ್ಲಿ          NOUN     KN_VERYLONG      B-LOC ◀ ENTITY
ಇರುವ                   NOUN     KN_MEDIUM        O
ಬಿಬಿಎಂಪಿ               NOUN     KN_LONG          B-ORG ◀ ENTITY
ಕಚೇರಿಗೆ                NOUN     KN_LONG          O
ಹೋಗಿ                   NOUN     KN_MEDIUM        O
ಸಚಿವರನ್ನು              NOUN     KN_LONG          O
ಭೇಟಿ                   NOUN     KN_MEDIUM        O
ಮಾಡಿದರು                VERB     KN_LONG          O



In [ ]:
predict_ner("ಬೆಂಗಳೂರಿನ ಇಂದಿರಾನಗರದಲ್ಲಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.")
print()


Input : ಬೆಂಗಳೂರಿನ ಇಂದಿರಾನಗರದಲ್ಲಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.

Token                  POS      Shape            Predicted NER
----------------------------------------------------------------------
ಬೆಂಗಳೂರಿನ              NOUN     KN_LONG          B-LOC ◀ ENTITY
ಇಂದಿರಾನಗರದಲ್ಲಿ         NOUN     KN_VERYLONG      B-LOC ◀ ENTITY
ಸುಧಾ                   NOUN     KN_MEDIUM        O
ಅವರ                    NOUN     KN_MEDIUM        O
ಮನೆಗೆ                  NOUN     KN_MEDIUM        O
ನುಗ್ಗಿ                 NOUN     KN_MEDIUM        O
ಕಳ್ಳರು                 NOUN     KN_MEDIUM        O
ದರೋಡೆ                  NOUN     KN_MEDIUM        O
ಮಾಡಿದ್ದಾರೆ.            NOUN     KN_LONG          O
ನಿವಾಸಿ                 NOUN     KN_MEDIUM        O
ಮಹೇಶ್                  NOUN     KN_MEDIUM        B-PER ◀ ENTITY
ಅವರು                   PART     KN_MEDIUM        O
ಪೊಲೀಸ್                 NOUN     KN_MEDIUM        O
ಠಾಣೆಗೆ                 NOUN     KN_M

In [ ]:
predict_ner("ಬೆಂಗಳೂರಿನ ಇಂದಿರಾ ಅವರ ತಾಯಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ಮಾರತಹಳ್ಳಿ ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.")
print()


Input : ಬೆಂಗಳೂರಿನ ಇಂದಿರಾ ಅವರ ತಾಯಿ ಸುಧಾ ಅವರ ಮನೆಗೆ ನುಗ್ಗಿ ಕಳ್ಳರು ದರೋಡೆ ಮಾಡಿದ್ದಾರೆ. ಮಾರತಹಳ್ಳಿ ನಿವಾಸಿ ಮಹೇಶ್ ಅವರು ಪೊಲೀಸ್ ಠಾಣೆಗೆ ದೂರು ನೀಡಿದರು.

Token                  POS      Shape            Predicted NER
----------------------------------------------------------------------
ಬೆಂಗಳೂರಿನ              NOUN     KN_LONG          B-LOC ◀ ENTITY
ಇಂದಿರಾ                 NOUN     KN_MEDIUM        B-PER ◀ ENTITY
ಅವರ                    NOUN     KN_MEDIUM        O
ತಾಯಿ                   NOUN     KN_MEDIUM        O
ಸುಧಾ                   NOUN     KN_MEDIUM        B-PER ◀ ENTITY
ಅವರ                    NOUN     KN_MEDIUM        O
ಮನೆಗೆ                  NOUN     KN_MEDIUM        O
ನುಗ್ಗಿ                 NOUN     KN_MEDIUM        O
ಕಳ್ಳರು                 NOUN     KN_MEDIUM        O
ದರೋಡೆ                  NOUN     KN_MEDIUM        O
ಮಾಡಿದ್ದಾರೆ.            NOUN     KN_LONG          O
ಮಾರತಹಳ್ಳಿ              NOUN     KN_LONG          O
ನಿವಾಸಿ                 NOUN     KN_MEDIUM        O
ಮಹೇಶ್                  NO

In [ ]:
predict_ner("ಬೆಸ್ಕಾಂ ಅಧಿಕಾರಿಗಳು ಮೂರು ದಿನಗಳಿಂದ ವಿದ್ಯುತ್ ಸಮಸ್ಯೆ ಬಗೆಹರಿಸಿಲ್ಲ. ಗ್ರಾಹಕ ಶಂಕರ್ ಅವರು ದೂರು ನೀಡಿದರು.")
print()


Input : ಬೆಸ್ಕಾಂ ಅಧಿಕಾರಿಗಳು ಮೂರು ದಿನಗಳಿಂದ ವಿದ್ಯುತ್ ಸಮಸ್ಯೆ ಬಗೆಹರಿಸಿಲ್ಲ. ಗ್ರಾಹಕ ಶಂಕರ್ ಅವರು ದೂರು ನೀಡಿದರು.

Token                  POS      Shape            Predicted NER
----------------------------------------------------------------------
ಬೆಸ್ಕಾಂ                NOUN     KN_LONG          B-LOC ◀ ENTITY
ಅಧಿಕಾರಿಗಳು             NOUN     KN_LONG          O
ಮೂರು                   NOUN     KN_MEDIUM        O
ದಿನಗಳಿಂದ               NOUN     KN_LONG          O
ವಿದ್ಯುತ್               NOUN     KN_LONG          O
ಸಮಸ್ಯೆ                 NOUN     KN_MEDIUM        O
ಬಗೆಹರಿಸಿಲ್ಲ.           NOUN     KN_VERYLONG      O
ಗ್ರಾಹಕ                 NOUN     KN_MEDIUM        B-PER ◀ ENTITY
ಶಂಕರ್                  NOUN     KN_MEDIUM        I-PER ◀ ENTITY
ಅವರು                   PART     KN_MEDIUM        O
ದೂರು                   NOUN     KN_MEDIUM        O
ನೀಡಿದರು.               NOUN     KN_LONG          O

